# Adult Income ML Pipeline
**Praful Chunchu** | AIPI 510 | Week 12 Interactive

Simple ML pipeline that predicts whether a person earns >$50K using the Adult Income dataset (OpenML, version 2) and logistic regression.

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [2]:
# load adult income dataset (version 2 pinned for reproducibility)
adult = fetch_openml(name="adult", version=2, as_frame=True)
income_df = adult.frame

In [3]:
# first look
income_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
# summary stats
income_df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [5]:
# check dtypes - numeric vs categorical
income_df.dtypes

age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
class             category
dtype: object

In [6]:
income_df.shape

(48842, 15)

In [7]:
# missing values - only workclass, occupation and native-country
income_df.isna().sum()

age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
class                0
dtype: int64

In [8]:
# education and education-num map 1:1, so we only need one
pd.crosstab(index=income_df['education-num'], columns=income_df['education'])

education,10th,11th,12th,1st-4th,5th-6th,7th-8th,9th,Assoc-acdm,Assoc-voc,Bachelors,Doctorate,HS-grad,Masters,Preschool,Prof-school,Some-college
education-num,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,83,0,0
2,0,0,0,247,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,509,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,955,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,756,0,0,0,0,0,0,0,0,0
6,1389,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,1812,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,657,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,15784,0,0,0,0


In [9]:
# class balance (~76/24 so we stratify the split)
income_df['class'].value_counts(normalize=True)

class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64

In [10]:
# drop fnlwgt (census sampling weight) and education (duplicate of education-num)
income_df.drop(columns=(['education', 'fnlwgt']), inplace=True)

In [11]:
# separate features and target
X = income_df.drop(columns=['class'])
y = income_df['class']

# We split the data into train and temp (We will split this 20% of temp into validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
    

In [12]:
# split temp 50/50 into validation and test (80/10/10 overall)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size = 0.50,
    random_state = 42,
    stratify=y_temp
)

In [13]:
# check split sizes
print("X_train Shape: " + str(X_train.shape))
print("Y_train Shape: " + str(y_train.shape))
print("X_test Shape: " + str(X_test.shape))

print("X_val Shape: " + str(X_val.shape))
print("y_val Shape: " + str(y_val.shape))
print("y_test Shape: " + str(y_test.shape))

X_train Shape: (39073, 12)
Y_train Shape: (39073,)
X_test Shape: (4885, 12)
X_val Shape: (4884, 12)
y_val Shape: (4884,)
y_test Shape: (4885,)


In [14]:
# check stratification - all splits should be ~24% >50K
y_train.value_counts(normalize=True)

class
<=50K    0.76073
>50K     0.23927
Name: proportion, dtype: float64

In [15]:
y_val.value_counts(normalize=True)

class
<=50K    0.760647
>50K     0.239353
Name: proportion, dtype: float64

In [16]:
y_test.value_counts(normalize=True)

class
<=50K    0.760696
>50K     0.239304
Name: proportion, dtype: float64

In [17]:
# column lists for the ColumnTransformer (from X_train so the target isn't included)
numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()

In [18]:
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

In [19]:
numeric_cols

['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

In [20]:
categorical_cols

['workclass',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# numeric: fill blanks with median, then scale
numeric_pipe = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# categorical: fill blanks with "Missing" as its own category, then one-hot encode
cat_pipe = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# send each column type through its own pipeline
preprocessor = ColumnTransformer(
    transformers = [
        ("num", numeric_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ]
)

# full pipeline: preprocessing + logistic regression
pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression())]
)

In [22]:
# fit on train only
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'education-num',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 strategy='constant')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['workclass',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('classifier', LogisticRegression())])

In [23]:
# train vs val accuracy to check for overfitting
train_accuracy = pipeline.score(X_train, y_train)
val_accuracy = pipeline.score(X_val, y_val)

In [24]:
print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Val accuracy:   {val_accuracy:.4f}")

Train accuracy: 0.8525
Val accuracy:   0.8591


In [25]:
# Baseline: always predicts the majority class
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report


baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print(f"Baseline val accuracy: {baseline.score(X_val, y_val):.4f}")

Baseline val accuracy: 0.7606


In [26]:
# precision/recall/f1 on validation
y_val_pred = pipeline.predict(X_val)
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      3715
        >50K       0.76      0.61      0.67      1169

    accuracy                           0.86      4884
   macro avg       0.82      0.77      0.79      4884
weighted avg       0.85      0.86      0.85      4884



In [27]:
# final evaluation on test (run once)
test_accuracy = pipeline.score(X_test, y_test)
print(f"Test accuracy: {test_accuracy:.4f}")

y_test_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred))

Test accuracy: 0.8516
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91      3716
        >50K       0.74      0.59      0.66      1169

    accuracy                           0.85      4885
   macro avg       0.81      0.76      0.78      4885
weighted avg       0.84      0.85      0.85      4885



## Interpretation

The model reaches 85.2% test accuracy, about 9 points above the 76.1% baseline of always predicting `<=50K`, so it is learning real signal from the features. Train (85.3%), validation (85.9%) and test (85.2%) accuracy are all close, which means the model is not overfitting and the validation score was a reliable estimate.

Accuracy alone hides the weak spot. For the `>50K` class, recall is 0.59, so the model misses about 4 in 10 high earners and tends to default to the majority class. When it does predict `>50K` it is right 74% of the time. If catching more high earners mattered, a next step would be trying `class_weight="balanced"` on the validation set, which should raise recall at some cost to precision.

In [28]:
# save the fitted pipeline (preprocessing + model)
import joblib

joblib.dump(pipeline, "adult_income_pipeline.joblib")

['adult_income_pipeline.joblib']

In [29]:
# versions for requirements.txt
import sklearn, pandas, joblib
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pandas.__version__}")
print(f"joblib=={joblib.__version__}")

scikit-learn==1.6.1
pandas==2.3.3
joblib==1.5.3
